# Q-обучение

### Цель и задачи

Создать среду, в которой имеются агент-доставщик полезного груза по городу.


Для этого необходимо сформировать карту, в которой будет перемещаться агент. Пример карты для одного агента в файле “citymap.txt”. Минимальные размеры карты 30 x 30.

In [1]:
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from collections import defaultdict
import os

### Пояснения к карте

0 (а также 2, 3, 4) – области на карте, куда агент может попасть (дорога).

1 – на карте отмечены препятствия, куда агент не может попасть (стены). 2 – место, с которого агент начинает движение по карте.

3 – места, с которых с определенной вероятностью выходят другие транспортные средства (вероятность этого мала, например 0.02).

4 – Область, в которую должен попасть агент.


## Чтение карты

In [2]:
with open('citymap.txt', 'r', encoding='utf-8') as f:
    grid_str = [line.strip() for line in f.readlines() if line.strip()]

ROWS = len(grid_str)
COLS = len(grid_str[0]) if ROWS > 0 else 0

original_grid = np.array([list(line) for line in grid_str], dtype=str)

goal_pos = None
agent4_start = None
spawn_points = []

In [3]:
for r in range(ROWS):
    for c in range(COLS):
        if original_grid[r][c] == '4':
            goal_pos = (r, c)
        elif original_grid[r][c] == '2':
            agent4_start = (r, c)
        elif original_grid[r][c] == '3':
            spawn_points.append((r, c))

In [4]:
start_candidates = [agent4_start] + spawn_points
print(f"Найдено {len(start_candidates)} стартовых точек: {start_candidates}")
print(f"Цель: {goal_pos}")

walkable = np.isin(original_grid, ['0', '2', '3', '4'])

Найдено 11 стартовых точек: [(28, 1), (3, 9), (9, 17), (10, 6), (11, 28), (15, 6), (15, 20), (20, 20), (23, 28), (26, 9), (26, 16)]
Цель: (1, 28)


## Вспомогательные функции

In [5]:
directions = [(-1, 0), (1, 0), (0, -1), (0, 1), (0, 0)]

In [6]:
def is_valid(r, c, grid):
    return 0 <= r < ROWS and 0 <= c < COLS and grid[r][c] == '0'

In [7]:
def manhattan_distance(p1, p2):
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

## Агент машина 3 и состояние

In [8]:
class Agent3:
    def __init__(self, pos, lifetime):
        self.pos = pos
        self.lifetime = lifetime
        self.prev_dir = None

In [ ]:
def choose_direction_agent3(pos, prev_dir_idx, grid):
    r, c = pos
    valid_dirs = []
    for i, (dr, dc) in enumerate(directions[:-1]):
        nr, nc = r + dr, c + dc
        if is_valid(nr, nc, grid):
            valid_dirs.append(i)
    if not valid_dirs:
        return 4
    if prev_dir_idx is not None and prev_dir_idx < 4 and prev_dir_idx in valid_dirs and random.random() < 0.7:
        return prev_dir_idx
    return random.choice(valid_dirs)

In [10]:
def get_state_mask(agent_pos, agents3_positions, grid):
    r, c = agent_pos
    mask = 0
    neigh = [(-1,-1), (-1,0), (-1,1), (0,-1), (0,1), (1,-1), (1,0), (1,1)]
    for i, (dr, dc) in enumerate(neigh):
        nr, nc = r + dr, c + dc
        if (nr, nc) in agents3_positions:
            mask |= (1 << i)
    wall_dirs = [(-1,0), (1,0), (0,-1), (0,1)]
    for i, (dr, dc) in enumerate(wall_dirs):
        nr, nc = r + dr, c + dc
        if not is_valid(nr, nc, grid):
            mask |= (1 << (8 + i))
    return mask

STATE_SIZE = 4096

## Параметры

In [11]:
alpha = 0.3
gamma = 0.94
epsilon_start = 0.5
episodes = 10000
max_steps = 500
spawn_probs = [0.005, 0.01, 0.02, 0.04, 0.08]
target_success = 0.92

os.makedirs('results', exist_ok=True)

## Основной цикл

In [12]:
summary_data = []
success_step_histories = {}

for start_idx, agent4_start in enumerate(start_candidates):
    print(f"\n{'='*80}")
    print(f"Стартовая точка {start_idx+1}/{len(start_candidates)}: {agent4_start}")
    print(f"{'='*80}")

    grid = original_grid.copy()
    grid[~walkable] = '1'
    grid[walkable] = '0'
    grid[goal_pos[0], goal_pos[1]] = '0'

    current_spawn_points = [p for p in spawn_points if p != agent4_start]

    for spawn_prob in spawn_probs:
        print(f"\n" + "-"*60)
        print(f"Обучение: p = {spawn_prob:.3f} | Старт: {agent4_start}")
        print("-"*60)

        Q = np.zeros((ROWS, COLS, STATE_SIZE, 5))

        steps_per_episode = []
        success_steps = []
        success_count = 0
        best_steps = float('inf')
        best_history = []
        spawn_cooldown = max(1, int(15 / spawn_prob))
        spawn_timer = 0
        initial_spawn_done = False

        for ep in range(episodes):
            epsilon = max(0.03, epsilon_start * (1 - ep / episodes))
            agent4_pos = agent4_start
            agents3 = []
            step = 0
            done = False
            episode_history = []

            if not initial_spawn_done and ep == 0:
                for sp in current_spawn_points[:min(2, len(current_spawn_points))]:
                    agents3.append(Agent3(sp, random.randint(50, 200)))
                initial_spawn_done = True

            while not done and step < max_steps:
                step += 1
                current_agents3 = [a.pos for a in agents3]
                episode_history.append((agent4_pos, current_agents3.copy(), step, False))

                spawn_timer += 1
                if spawn_timer >= spawn_cooldown:
                    spawn_timer = 0
                    for sp in current_spawn_points:
                        if random.random() < 0.7:
                            agents3.append(Agent3(sp, random.randint(50, 300)))

                agents3_pos = {a.pos for a in agents3}
                state = get_state_mask(agent4_pos, agents3_pos, grid)
                r, c = agent4_pos
                action = random.randint(0, 4) if random.random() < epsilon else np.argmax(Q[r, c, state])
                dr, dc = directions[action]
                intended4 = (agent4_pos[0] + dr, agent4_pos[1] + dc)
                new_agent4_pos = intended4 if is_valid(*intended4, grid) else agent4_pos

                intended = [intended4]
                actions3 = [choose_direction_agent3(a.pos, a.prev_dir, grid) for a in agents3]
                for a, dir_idx in zip(agents3, actions3):
                    a.prev_dir = dir_idx
                    a.lifetime -= 1
                    dr, dc = directions[dir_idx]
                    nr, nc = a.pos[0] + dr, a.pos[1] + dc
                    intended.append((nr, nc) if is_valid(nr, nc, grid) else a.pos)

                pos_count = defaultdict(int)
                for pos in intended:
                    if is_valid(*pos, grid) or pos == agent4_pos:
                        pos_count[pos] += 1
                collision = any(v > 1 for v in pos_count.values())

                if collision:
                    reward = -150
                    done = True
                    episode_history[-1] = (agent4_pos, current_agents3, step, True)
                elif new_agent4_pos == goal_pos:
                    reward = 400
                    done = True
                    success_count += 1
                    success_steps.append(step)
                    episode_history.append((new_agent4_pos, [a.pos for a in agents3], step, False))
                    if step < best_steps:
                        best_steps = step
                        best_history = episode_history.copy()
                else:
                    dist = manhattan_distance(new_agent4_pos, goal_pos)
                    reward = 22 / (dist + 1) - 0.8
                    if any(manhattan_distance(new_agent4_pos, p) <= 1 for p in intended[1:]):
                        reward -= 20

                next_agents3_pos = {intended[i+1] for i in range(len(agents3))}
                next_state = get_state_mask(new_agent4_pos, next_agents3_pos, grid)
                Q[r, c, state, action] += alpha * (
                    reward + gamma * np.max(Q[new_agent4_pos[0], new_agent4_pos[1], next_state]) - Q[r, c, state, action]
                )

                agent4_pos = new_agent4_pos
                for i, a in enumerate(agents3):
                    a.pos = intended[i + 1]
                agents3 = [a for a in agents3 if a.lifetime > 0]

            steps_per_episode.append(step if step < max_steps else max_steps)

            rate = success_count / (ep + 1)
            if ep % 1000 == 0 and ep > 0:
                print(f" Эпизод {ep:5d} | Успех: {success_count:5d} ({rate:6.1%}) | Шаги: {np.mean(steps_per_episode[-1000:]):.1f}")

            if rate >= target_success and ep > 100:
                print(f" ДОСТИГНУТО {target_success:.0%}! Остановка.")
                break

        avg = np.mean(steps_per_episode)
        std = np.std(steps_per_episode)
        final_success = success_count / len(steps_per_episode) * 100
        print(f"→ Старт {agent4_start}, p={spawn_prob:.3f}: {avg:.1f} ± {std:.1f} шагов, Успех: {final_success:.1f}%")

        key = (agent4_start, spawn_prob)
        summary_data.append((agent4_start, spawn_prob, avg, std, final_success))
        success_step_histories[key] = success_steps

        np.save(f'results/Q_start_{agent4_start[0]}_{agent4_start[1]}_p_{spawn_prob:.3f}.npy', Q)

        if best_history:
            print(f" Создаём GIF (лучший путь: {best_steps} шагов)...")
            fig, ax = plt.subplots(figsize=(10, 8))
            ax.set_xlim(-0.5, COLS - 0.5)
            ax.set_ylim(-0.5, ROWS - 0.5)
            ax.set_aspect('equal')
            ax.invert_yaxis()

            for r in range(ROWS):
                for c in range(COLS):
                    if grid[r][c] == '1':
                        ax.add_patch(patches.Rectangle((c-0.5, r-0.5), 1, 1, facecolor='gray', edgecolor='none'))

            ax.add_patch(patches.Rectangle((goal_pos[1]-0.5, goal_pos[0]-0.5), 1, 1, facecolor='yellow', edgecolor='gold', linewidth=2))

            agent4 = patches.Circle((0,0), 0.4, facecolor='blue', edgecolor='darkblue', linewidth=2)
            ax.add_patch(agent4)
            path_line, = ax.plot([], [], 'b-', linewidth=2, alpha=0.7)
            path_pts = []
            agent3_patches = []

            def init():
                agent4.center = (agent4_start[1], agent4_start[0])
                path_line.set_data([], [])
                path_pts.clear()
                for p in agent3_patches:
                    p.remove()
                agent3_patches.clear()
                return [agent4, path_line]

            def animate(frame):
                pos4, pos3, step_num, crash = best_history[frame]
                path_pts.append(pos4)
                xs, ys = zip(*[(c, r) for r, c in path_pts])
                path_line.set_data(xs, ys)
                agent4.center = (pos4[1], pos4[0])
                agent4.set_facecolor('red' if crash else 'blue')
                for p in agent3_patches:
                    p.remove()
                agent3_patches.clear()
                for r, c in pos3:
                    circ = patches.Circle((c, r), 0.35, facecolor='green', edgecolor='darkgreen', alpha=0.8)
                    ax.add_patch(circ)
                    agent3_patches.append(circ)
                status = "АВАРИЯ!" if crash else ("ДОСТАВЛЕНО!" if pos4 == goal_pos else "")
                ax.set_title(f'Старт: {agent4_start} | p={spawn_prob:.3f} | Шаг {step_num} | {status}', fontsize=11)
                return [agent4, path_line] + agent3_patches

            anim = FuncAnimation(fig, animate, init_func=init, frames=len(best_history), interval=500, blit=False)
            gif_path = f'results/best_path_start_{agent4_start[0]}_{agent4_start[1]}_p_{spawn_prob:.3f}.gif'
            anim.save(gif_path, writer='pillow', fps=2)
            plt.close(fig)
            print(f" GIF: {gif_path}")


Стартовая точка 1/11: (28, 1)

------------------------------------------------------------
Обучение: p = 0.005 | Старт: (28, 1)
------------------------------------------------------------
 ДОСТИГНУТО 92%! Остановка.
→ Старт (28, 1), p=0.005: 132.6 ± 90.2 шагов, Успех: 92.0%
 Создаём GIF (лучший путь: 71 шагов)...
 GIF: results/best_path_start_28_1_p_0.005.gif

------------------------------------------------------------
Обучение: p = 0.010 | Старт: (28, 1)
------------------------------------------------------------
 Эпизод  1000 | Успех:   738 ( 73.7%) | Шаги: 194.5
 Эпизод  2000 | Успех:  1687 ( 84.3%) | Шаги: 103.9
 Эпизод  3000 | Успех:  2637 ( 87.9%) | Шаги: 98.1
 Эпизод  4000 | Успех:  3597 ( 89.9%) | Шаги: 85.6
 Эпизод  5000 | Успех:  4556 ( 91.1%) | Шаги: 86.1
 Эпизод  6000 | Успех:  5520 ( 92.0%) | Шаги: 79.0
 ДОСТИГНУТО 92%! Остановка.
→ Старт (28, 1), p=0.010: 107.8 ± 81.8 шагов, Успех: 92.0%
 Создаём GIF (лучший путь: 56 шагов)...
 GIF: results/best_path_start_28_1_p_0.0

## Графики

ГРАФИК 1: Успех + шаги по p для каждого старта

In [13]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(start_candidates)))

for idx, start_pos in enumerate(start_candidates):
    probs = []
    avgs = []
    stds = []
    successes = []
    for p in spawn_probs:
        for data in summary_data:
            if data[0] == start_pos and data[1] == p:
                probs.append(p)
                avgs.append(data[2])
                stds.append(data[3])
                successes.append(data[4])
                break

    label = f'Старт ({start_pos[0]},{start_pos[1]})'
    ax1.plot(probs, avgs, 'o-', color=colors[idx], label=label)
    ax1.fill_between(probs, [a-s for a,s in zip(avgs, stds)], [a+s for a,s in zip(avgs, stds)], color=colors[idx], alpha=0.2)
    ax2.plot(probs, successes, 's-', color=colors[idx], marker='D', label=label)

ax1.set_title('Среднее число шагов до завершения')
ax1.set_xlabel('Вероятность спавна (p)')
ax1.set_ylabel('Шаги')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.set_title('Успешные доставки, %')
ax2.set_xlabel('Вероятность спавна (p)')
ax2.set_ylabel('Успех, %')
ax2.set_ylim(0, 100)
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.savefig('results/summary_by_start.png', dpi=300)
plt.close()



C:\Users\user\AppData\Local\Temp\ipykernel_23060\1520425507.py:21: UserWarning: marker is redundantly defined by the 'marker' keyword argument and the fmt string "s-" (-> marker='s'). The keyword argument will take precedence.
  ax2.plot(probs, successes, 's-', color=colors[idx], marker='D', label=label)


ГРАФИК 2: Распределение шагов при успехе

In [14]:
plt.figure(figsize=(12, 7))
bins = np.arange(0, max_steps + 20, 20)
for idx, start_pos in enumerate(start_candidates):
    for prob in spawn_probs:
        key = (start_pos, prob)
        if key in success_step_histories and success_step_histories[key]:
            hist, _ = np.histogram(success_step_histories[key], bins=bins, density=True)
            bin_centers = (bins[:-1] + bins[1:]) / 2
            plt.plot(bin_centers, hist, 'o-', label=f'Старт {start_pos}, p={prob:.3f}')

plt.title('Распределение шагов до цели (только успехи)')
plt.xlabel('Количество шагов')
plt.ylabel('Плотность вероятности')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/success_step_distribution_by_start.png', dpi=300, bbox_inches='tight')
plt.close()